# **1. Perkenalan Dataset**

Dataset ini adalah dataset Titanic, yang diambil dari public repository. Dataset ini digunakan untuk memprediksi probabilitas keselamatan (survival) dari penumpang kapal Titanic berdasarkan berbagai fitur seperti umur, jenis kelamin, kelas tiket, dan lain-lain.

# **2. Import Library**
Pada tahap ini, kita mengimpor pustaka yang dibutuhkan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os

# **3. Memuat Dataset**
Memuat dataset titanic_raw.csv.

In [ ]:
df = pd.read_csv("../titanic_raw.csv")
df.head()

# **4. Exploratory Data Analysis (EDA)**
Melakukan eksplorasi awal pada data.

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())

sns.countplot(x="Survived", hue="Sex", data=df)
plt.title("Survival by Gender")
plt.show()

# **5. Data Preprocessing**
Membersihkan nilai kosong dan melakukan encoding pada fitur kategorikal.

In [ ]:
# Drop irrelevant columns
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Fill missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Encode categorical features
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

# Save preprocessed data
output_dir = 'titanic_preprocessing'
os.makedirs(output_dir, exist_ok=True)
df.to_csv(f'{output_dir}/titanic_clean.csv', index=False)
df.head()

# **6. Model Training dan MLflow Tracking**
Melatih model Random Forest dan melakukan tracking eksperimen menggunakan MLflow.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Enable MLflow autologging
mlflow.sklearn.autolog()

X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name='Eksperimen_Dhafin_Lubis'):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {acc}')
